# Phase 2 · Legal Named Entity Recognition — Knowledge distillation: Architecture-4 teacher (partial FT) → Architecture-2 encoder student

| | |
|---|---|
| **Codabench** | Strict Macro-F1 **0.4445** · Strict Macro-P 0.4917 · Strict Macro-R 0.436 (rank 3 of 4 in Phase 2) |
| **Own dev / validation** | fold-2 strict macro-F1: teacher 0.3618, student 0.4309 |
| **Backbone** | Architecture-2 encoder-decoder (150M) |

**The task, in simple words:** find the important names in a judgment — appellant, respondent, judge, court, statute, precedent, date, … — as exact character positions with a label.

**How this notebook solves it:**
1. Split every document into words, remembering their exact character positions.
2. Read the words with the Phase-1 model, in overlapping windows (documents are longer than the model's context).
3. Give every word a tag: **B** = begins an entity, **I** = inside one, **O** = outside; the head makes sure the tag sequence is valid.
4. Turn the tags back into `[start, end, label]` spans.
5. Train on fold 1 and check on fold 2; then retrain on both folds and predict the test fold.

**What is special in this run:** **Knowledge distillation** — a larger / stronger *teacher* is trained first; a smaller *student* then learns from the gold labels **and** from the teacher's soft predictions, to keep most of the quality at a lower cost.

*Exact setting:* teacher: top 8 of 16 Architecture-4 blocks fine-tuned (layer-wise LR decay 0.85); student: full Architecture-2 encoder.

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoints (teacher and student) as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [19]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [20]:
# ============================================================
# CELL 2 — CONFIGURATION
#   E4: Architecture-4 hybrid  = TEACHER  (partial / layer-wise fine-tuning)
#   E5: Architecture-2 encoder = STUDENT  (knowledge distillation from E4)
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 checkpoints. If a path is wrong, CELL 11 searches /kaggle/input for it.
TEACHER_CHECKPOINT = Path(       # Architecture-4 (HybridLM) best.pt
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-4/pytorch/default/1/best.pt"
)
STUDENT_CHECKPOINT = Path(       # Architecture-2 (encoder-decoder) best_encoder_decoder.pt
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-2/pytorch/default/1/best_encoder_decoder.pt"
)

OUTPUT_DIR = Path("/kaggle/working/e4_e5_teacher_student")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FOLDS = ["fold1"]          # stage A: train
VAL_FOLDS = ["fold2"]            # stage A: validate (scores for the report)

# Stage B: retrain teacher and student on fold1 + fold2 (best epochs of stage A)
# and write the two test submissions (E4 and E5).
FINAL_TRAIN_ON_ALL_FOLDS = True

WINDOW = 512
STRIDE = 384


# ------------------------------------------------------------
# Head (same as E1 / E2)
# ------------------------------------------------------------

SHAPE_DIM = 32
LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.3


# ------------------------------------------------------------
# E4 TEACHER: partial / layer-wise fine-tuning of Architecture-4
# (embeddings + lower blocks frozen, top blocks + head trained)
# ------------------------------------------------------------

TEACHER_TRAIN_TOP_BLOCKS = 8     # of 16 blocks (4 was too few: the teacher underfitted)
TEACHER_EPOCHS = 20
TEACHER_PATIENCE = 5
TEACHER_BATCH_SIZE = 8           # if CUDA out-of-memory: 4
LR_TEACHER_BLOCKS = 3e-4
LR_TEACHER_HEAD = 2e-3
TEACHER_LAYER_LR_DECAY = 0.85    # layer-wise LR: top block LR_TEACHER_BLOCKS, each lower block x 0.85


# ------------------------------------------------------------
# E5 STUDENT: Architecture-2 encoder, full fine-tuning + distillation
#   loss = (1 - KD_ALPHA) * CRF loss on gold tags
#        + KD_ALPHA * T^2 * KL(teacher || student)   (temperature T)
# ------------------------------------------------------------

KD_ALPHA = 0.5
KD_TEMPERATURE = 2.0

# Selective distillation: use the teacher's soft targets only on words the
# teacher tags correctly, so the student does not copy the teacher's mistakes
# (e.g. a teacher that never finds APP / RESP / WIT would teach the student "O").
KD_ONLY_WHERE_TEACHER_CORRECT = True

STUDENT_EPOCHS = 30
STUDENT_PATIENCE = 6
STUDENT_BATCH_SIZE = 8
LR_STUDENT_ENCODER = 1e-4
LR_STUDENT_HEAD = 2e-3


# ------------------------------------------------------------
# Shared training settings
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 16
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
TRAIN_EVAL_DOCS = 8

# Document-level consistency after decoding: a name tagged once is tagged
# everywhere in its document (kept only for labels where it helps on validation).
DOC_PROPAGATION = True


# ------------------------------------------------------------
# Debug mode
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 12

if DEBUG_MODE:
    TEACHER_EPOCHS = STUDENT_EPOCHS = 2


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


print("Stage A:", TRAIN_FOLDS, "->", VAL_FOLDS, "| Stage B (all folds):", FINAL_TRAIN_ON_ALL_FOLDS)
print(f"Teacher: top {TEACHER_TRAIN_TOP_BLOCKS} blocks trained | KD alpha {KD_ALPHA}, temperature {KD_TEMPERATURE}")
print("AMP dtype:", AMP_DTYPE if USE_AMP else "disabled")

Stage A: ['fold1'] -> ['fold2'] | Stage B (all folds): True
Teacher: top 8 blocks trained | KD alpha 0.5, temperature 2.0
AMP dtype: torch.float16


### CELL 3 · Load tokenizer + tokenize with character offsets

**In simple words:** NER answers must be exact character positions, so while tokenizing we also remember where every token starts and ends in the original text.

**How it works:** Same pre-tokenizer and BPE merges as `tokenizer.py`, plus a `(start, end)` character offset for every token.

In [21]:
# ============================================================
# CELL 3 — LOAD TOKENIZER + TOKENIZE WITH CHARACTER OFFSETS
#
# NER must return CHARACTER positions, so for every token we
# also need the exact start / end character in the ORIGINAL text.
# We use the same pre-tokenizer and BPE as tokenizer.py, but
# keep track of positions.
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


bpe_cache = {}      # piece -> list of BPE tokens (big speed-up: words repeat a lot)


def bpe_tokens(piece):

    if piece not in bpe_cache:
        bpe_cache[piece] = tokenizer._apply_bpe(list(piece))

    return bpe_cache[piece]


def encode_with_offsets(text):
    """
    Returns four lists (one entry per token):
        ids    - token ids (identical to tokenizer.encode(text))
        starts - start character in the ORIGINAL text
        ends   - end character (exclusive) in the ORIGINAL text
        words  - index of the word (pre-tokenizer piece) the token belongs to
    """

    normalized = tok_module.normalize_text(text)

    if normalized == text:

        work_text = text          # normal case: positions are unchanged
        origin = None

    else:

        # NFKC can change the text length (for example "ﬁ" -> "fi").
        # Normalize character by character and remember where each
        # character came from.
        chars = []
        origin = []

        for i, ch in enumerate(text):
            n = tok_module.normalize_text(ch)
            chars.append(n)
            origin.extend([i] * len(n))

        work_text = "".join(chars)

    ids, starts, ends, words = [], [], [], []

    for word_index, match in enumerate(tok_module.PRETOKEN_PATTERN.finditer(work_text)):

        position = match.start()

        for token in bpe_tokens(match.group()):

            token_id = tokenizer.vocab.get(token)

            # (token, start in work_text) pieces; unknown tokens fall back to characters
            if token_id is not None:
                parts = [(token_id, position, position + len(token))]
            else:
                parts = [
                    (tokenizer.vocab.get(ch, UNK_ID), position + k, position + k + 1)
                    for k, ch in enumerate(token)
                ]

            for part_id, s, e in parts:

                ids.append(part_id)
                words.append(word_index)

                if origin is None:
                    starts.append(s)
                    ends.append(e)
                else:
                    starts.append(origin[s])
                    ends.append(origin[e - 1] + 1)

            position += len(token)

    return ids, starts, ends, words


# ------------------------------------------------------------
# Sanity test
# ------------------------------------------------------------

demo = "The Supreme Court of India held that Section 12 applies."

demo_ids, demo_s, demo_e, demo_w = encode_with_offsets(demo)

assert demo_ids == tokenizer.encode(demo)

print("\nToken           start end  word  text")

for i in range(min(10, len(demo_ids))):
    print(f"  {tokenizer.id_to_token[demo_ids[i]]!r:14} {demo_s[i]:4d} {demo_e[i]:4d} {demo_w[i]:4d}  {demo[demo_s[i]:demo_e[i]]!r}")

Vocabulary size: 32004 | PAD / UNK: 0 1

Token           start end  word  text
  'The'             0    3    0  'The'
  ' '               3    4    1  ' '
  'Supreme'         4   11    2  'Supreme'
  ' '              11   12    3  ' '
  'Court'          12   17    4  'Court'
  ' '              17   18    5  ' '
  'of'             18   20    6  'of'
  ' '              20   21    7  ' '
  'India'          21   26    8  'India'
  ' '              26   27    9  ' '


### CELL 4 · Download Phase-2 data (needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [22]:
# ============================================================
# CELL 4 — DOWNLOAD PHASE-2 DATA  (needs the Kaggle secret "HF")
# ============================================================

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")

PHASE2_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels":     "Phase_2_LNER/Train/labels.txt",
    "test_text":  "Phase_2_LNER/Test/fold3-text.json",
}

DATA_FILES = {}

for name, filename in PHASE2_FILES.items():

    DATA_FILES[name] = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    print(f"{name:11s} -> {DATA_FILES[name]}")

fold1_text  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
fold1_gold  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
fold2_text  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
fold2_gold  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
labels      -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
test_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Ph

### CELL 5 · Load texts, labels and gold entities

**In simple words:** Read the documents of fold 1, fold 2 and the test fold, the 12 entity labels, and the gold entities.

**How it works:** Every gold entity is `(start, end, label)` with end-exclusive character offsets.

In [23]:
# ============================================================
# CELL 5 — LOAD TEXTS, LABELS AND GOLD ENTITIES
# ============================================================

def load_texts(path):
    """{doc_id: text}"""

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, dict), "expected {doc_id: text}"

    return {
        str(doc_id): (value["text"] if isinstance(value, dict) else value)
        for doc_id, value in data.items()
    }


with open(DATA_FILES["labels"], "r", encoding="utf-8") as f:
    LABELS = [line.strip() for line in f if line.strip()]

print(f"{len(LABELS)} labels:", LABELS)


def parse_annotation(annotation):
    """One gold entity -> (start, end, label, entity_text or None)."""

    if isinstance(annotation, dict):

        def pick(*names):
            for name in names:
                if name in annotation:
                    return annotation[name]
            raise KeyError(f"none of {names} in {annotation}")

        return (
            int(pick("start", "start_char", "begin")),
            int(pick("end", "end_char")),
            str(pick("label", "type", "tag")),
            annotation.get("text", annotation.get("entity", annotation.get("entity_text"))),
        )

    # list / tuple: [start, end, label, (entity_text)]
    entity_text = annotation[3] if len(annotation) > 3 else None

    return int(annotation[0]), int(annotation[1]), str(annotation[2]), entity_text


def load_gold(path, texts):
    """{doc_id: [[start, end, label], ...]} with end EXCLUSIVE."""

    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    assert isinstance(raw, dict), f"expected {{doc_id: [entities]}}, got {type(raw)}"

    parsed = {}

    for doc_id, annotations in raw.items():

        if isinstance(annotations, dict):
            annotations = annotations.get("entities", annotations.get("annotations", []))

        parsed[str(doc_id)] = [parse_annotation(a) for a in annotations]

    # ---- is the end offset exclusive (Python style) or inclusive? ----
    exclusive = inclusive = checked = 0

    for doc_id, entities in parsed.items():

        text = texts.get(doc_id)

        for start, end, label, entity_text in entities:

            if text is None or entity_text is None:
                continue

            checked += 1
            exclusive += text[start:end] == entity_text
            inclusive += text[start:end + 1] == entity_text

    shift = 0

    if checked == 0:
        print("  WARNING: gold has no entity text, cannot check the end-offset convention (assuming exclusive).")
    else:
        print(f"  end-offset check on {checked} entities: exclusive matches={exclusive}, inclusive matches={inclusive}")

        if inclusive > exclusive:
            shift = 1
            print("  -> end offsets are INCLUSIVE in the gold file; converting to exclusive.")

    gold = {}
    unknown_labels = Counter()

    for doc_id, entities in parsed.items():

        gold[doc_id] = []

        for start, end, label, entity_text in entities:

            if label not in LABELS:
                unknown_labels[label] += 1
                continue

            gold[doc_id].append([start, end + shift, label])

    if unknown_labels:
        print("  WARNING: labels not in labels.txt were ignored:", dict(unknown_labels))

    return gold, shift


texts = {}
gold = {}
END_SHIFT = 0

for fold in sorted(set(TRAIN_FOLDS + VAL_FOLDS)):

    print(f"\n{fold}")

    texts[fold] = load_texts(DATA_FILES[f"{fold}_text"])
    gold[fold], shift = load_gold(DATA_FILES[f"{fold}_gold"], texts[fold])

    END_SHIFT = max(END_SHIFT, shift)

    missing = set(texts[fold]) - set(gold[fold])
    print(f"  documents: {len(texts[fold])} | entities: {sum(len(v) for v in gold[fold].values())} | "
          f"docs without gold entry: {len(missing)}")

    for doc_id in missing:
        gold[fold][doc_id] = []

    if DEBUG_MODE:
        keep = list(texts[fold])[:DEBUG_DOCS]
        texts[fold] = {k: texts[fold][k] for k in keep}
        gold[fold] = {k: gold[fold][k] for k in keep}


test_texts = load_texts(DATA_FILES["test_text"])

if DEBUG_MODE:
    test_texts = {k: test_texts[k] for k in list(test_texts)[:DEBUG_DOCS]}

print(f"\nTest documents: {len(test_texts)}")

# sample so you can see the real format
first_fold = TRAIN_FOLDS[0]
sample_id = next(iter(gold[first_fold]))
sample_text = texts[first_fold][sample_id]

print("\nSample document", sample_id, "->", repr(sample_text[:120]))

for start, end, label in gold[first_fold][sample_id][:5]:
    print(f"  [{start}, {end}, {label}]  {sample_text[start:end]!r}")

12 labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']

fold1
  end-offset check on 4120 entities: exclusive matches=4120, inclusive matches=9
  documents: 35 | entities: 4120 | docs without gold entry: 0

fold2
  end-offset check on 4273 entities: exclusive matches=4273, inclusive matches=8
  documents: 35 | entities: 4273 | docs without gold entry: 0

Test documents: 35

  [137, 153, RESP]  'STATE OF HARYANA'
  [252, 261, DATE]  '19.8.2011'
  [276, 308, COURT]  'High Court of Punjab and Haryana'
  [359, 369, COURT]  'High Court'
  [575, 585, DATE]  '18.11.1994'


### CELL 6 · Tags (BIO), word features, spans, strict F1

**In simple words:** Turn entities into one tag per word — **B** = first word of an entity, **I** = inside, **O** = outside — and back. Also simple word features (capital letters, digits, punctuation) and the scoring rule.

**How it works:** 12 labels → 25 BIO tags. Word shape + 6 flags help with names and numbers. Strict F1 = start, end **and** label must all match (the Codabench rule), averaged over the labels (macro-F1). Viterbi decoding is defined here too.

In [24]:
# ============================================================
# CELL 6 — TAGS (BIO), WORD FEATURES, SPANS, STRICT F1
#
# Every WORD gets one of 25 tags:  O, B-<label>, I-<label>
#     B = first word of an entity   I = inside an entity
# ============================================================

TAGS = ["O"]

for label in LABELS:
    TAGS += [f"B-{label}", f"I-{label}"]

TAG2ID = {tag: i for i, tag in enumerate(TAGS)}
NUM_TAGS = len(TAGS)

# tag id -> ("B" / "I" / "O", label)
TAG_PARTS = [("O", None)] + [(tag[0], tag[2:]) for tag in TAGS[1:]]

# valid transitions: "I-X" may only follow "B-X" or "I-X"; a sequence cannot start with "I-X"
ALLOWED = np.zeros((NUM_TAGS, NUM_TAGS), dtype=bool)          # [previous tag, current tag]

for prev in range(NUM_TAGS):
    for cur in range(NUM_TAGS):
        prev_prefix, prev_label = TAG_PARTS[prev]
        cur_prefix, cur_label = TAG_PARTS[cur]
        ALLOWED[prev, cur] = cur_prefix != "I" or (prev_prefix != "O" and prev_label == cur_label)

START_ALLOWED = np.array([TAG_PARTS[t][0] != "I" for t in range(NUM_TAGS)])

print("Number of tags:", NUM_TAGS)


# ------------------------------------------------------------
# Word features: shape + 6 simple flags. They generalise across
# documents (a new name is still "Xxxx Xxxx", a date "dd.dd.dddd").
# ------------------------------------------------------------

def word_shape(word):

    shape = []

    for ch in word[:20]:
        if ch.isupper():
            c = "X"
        elif ch.islower():
            c = "x"
        elif ch.isdigit():
            c = "d"
        else:
            c = ch

        if len(shape) < 2 or not (shape[-1] == c and shape[-2] == c):   # "Xxxxxx" -> "Xxx"
            shape.append(c)

    return "".join(shape)


def word_flags(word):

    return [
        float(word.istitle()),
        float(word.isupper()),
        float(word.islower()),
        float(any(ch.isdigit() for ch in word)),
        float(word.isdigit()),
        float(not any(ch.isalnum() for ch in word)),
    ]


NUM_FLAGS = 6


# ------------------------------------------------------------
# Gold spans <-> word tags
# ------------------------------------------------------------

def first_non_space(text, start, end):

    while start < end and text[start].isspace():
        start += 1

    return start


def spans_to_tags(spans, text, starts, ends):
    """Gold entities -> one tag id per word."""

    n = len(starts)

    tags = np.zeros(n, dtype=np.int64)               # all "O"

    ends_arr = np.asarray(ends)
    trimmed_starts = np.array([first_non_space(text, s, e) for s, e in zip(starts, ends)], dtype=np.int64)

    dropped = 0

    for ent_start, ent_end, label in sorted(spans):

        first = int(np.searchsorted(ends_arr, ent_start, side="right"))
        last = int(np.searchsorted(trimmed_starts, ent_end, side="left")) - 1

        if last < first or first >= n or np.any(tags[first:last + 1] != 0):
            dropped += 1
            continue

        tags[first] = TAG2ID[f"B-{label}"]
        tags[first + 1:last + 1] = TAG2ID[f"I-{label}"]

    return tags, dropped


def tags_to_spans(tag_ids, text, starts, ends):
    """One tag per word -> [[start, end, label], ...] in original characters."""

    spans = []
    current_label = None
    current_start = current_end = 0

    def finish():

        if current_label is None:
            return

        s, e = current_start, current_end

        while s < e and text[s].isspace():
            s += 1

        while e > s and text[e - 1].isspace():
            e -= 1

        if e > s:
            spans.append([s, e, current_label])

    for i, tag_id in enumerate(tag_ids):

        prefix, label = TAG_PARTS[int(tag_id)]

        if prefix == "O":
            finish()
            current_label = None

        elif prefix == "B" or label != current_label:
            finish()
            current_label = label
            current_start, current_end = starts[i], ends[i]

        else:
            current_end = ends[i]

    finish()

    return spans


# ------------------------------------------------------------
# Viterbi: best valid tag sequence for a whole document
# ------------------------------------------------------------

def viterbi(emissions, transitions, start_scores, end_scores):
    """emissions [n_words, NUM_TAGS] -> best tag id per word (numpy)."""

    n = len(emissions)

    if n == 0:
        return np.zeros(0, dtype=np.int64)

    score = start_scores + emissions[0]
    backpointer = np.zeros((n, NUM_TAGS), dtype=np.int64)

    for i in range(1, n):
        candidates = score[:, None] + transitions          # [previous, current]
        backpointer[i] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[i]

    path = [int((score + end_scores).argmax())]

    for i in range(n - 1, 0, -1):
        path.append(int(backpointer[i, path[-1]]))

    return np.array(path[::-1], dtype=np.int64)


# ------------------------------------------------------------
# Strict macro P / R / F1 (start, end and label must all match)
# ------------------------------------------------------------

def strict_scores(gold_spans, predicted_spans):

    tp, fp, fn = Counter(), Counter(), Counter()

    for doc_id in gold_spans:

        gold_set = {(s, e, l) for s, e, l in gold_spans[doc_id]}
        pred_set = {(s, e, l) for s, e, l in predicted_spans.get(doc_id, [])}

        for _, _, label in gold_set & pred_set:
            tp[label] += 1
        for _, _, label in pred_set - gold_set:
            fp[label] += 1
        for _, _, label in gold_set - pred_set:
            fn[label] += 1

    per_label = {}

    for label in LABELS:

        if tp[label] + fp[label] + fn[label] == 0:
            continue

        precision = tp[label] / (tp[label] + fp[label]) if tp[label] + fp[label] else 0.0
        recall = tp[label] / (tp[label] + fn[label]) if tp[label] + fn[label] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        per_label[label] = {"precision": precision, "recall": recall, "f1": f1,
                            "support": tp[label] + fn[label]}

    macro = {
        key: float(np.mean([v[key] for v in per_label.values()])) if per_label else 0.0
        for key in ("precision", "recall", "f1")
    }

    return macro, per_label

Number of tags: 25


### CELL 7 · Tokenize all documents (tokens -> words) + windows

**In simple words:** Tokenize every document, remember which tokens belong to which word, and cut long documents into overlapping windows that fit the model.

**How it works:** Every word is predicted from its first token; a word inside several windows gets the average of their scores.

In [25]:
# ============================================================
# CELL 7 — TOKENIZE ALL DOCUMENTS (tokens -> words) + WINDOWS
# ============================================================

def make_doc(doc_id, text, spans=None):

    ids, starts, ends, words = encode_with_offsets(text)

    # A new "word" (tagging unit) starts at a token when
    #   - a new pre-tokenizer piece starts, or
    #   - letters meet digits inside a piece  ("Verma1" -> Verma | 1, "dated24" -> dated | 24)
    #   - the token starts with an apostrophe ("Board's" -> Board | 's)
    # because gold entities often end exactly there.
    #   - whitespace-only tokens (" ", "\n") are NOT words: the model still reads
    #     them, but they are never tagged, so an entity can never break on a space
    is_space = np.array([text[s:e].strip() == "" for s, e in zip(starts, ends)], dtype=bool)

    new_word = np.zeros(len(ids), dtype=bool)

    for i in range(len(ids)):

        if is_space[i]:
            continue

        if i == 0 or is_space[i - 1] or words[i] != words[i - 1]:
            new_word[i] = True
        else:
            previous_char = text[ends[i - 1] - 1]
            current_char = text[starts[i]]
            new_word[i] = (previous_char.isdigit() != current_char.isdigit()) or current_char in "'’"

    first_token = np.flatnonzero(new_word)

    # last token of every word = last non-space token before the next word starts
    last_token = first_token.copy()

    for k in range(len(first_token)):
        j = first_token[k]
        while j + 1 < len(ids) and not new_word[j + 1] and not is_space[j + 1]:
            j += 1
        last_token[k] = j

    word_starts = [starts[i] for i in first_token]
    word_ends = [ends[i] for i in last_token]
    word_texts = [text[first_non_space(text, s, e):e] for s, e in zip(word_starts, word_ends)]

    doc = {
        "id": doc_id,
        "text": text,
        "ids": np.asarray(ids, dtype=np.int64),
        "first_token": first_token,
        "word_starts": word_starts,
        "word_ends": word_ends,
        "shapes": [word_shape(w) for w in word_texts],
        "flags": np.asarray([word_flags(w) for w in word_texts], dtype=np.float32).reshape(-1, NUM_FLAGS),
        "spans": spans,
        "word_tags": None,
        "dropped": 0,
    }

    if spans is not None:
        doc["word_tags"], doc["dropped"] = spans_to_tags(spans, text, word_starts, word_ends)

    return doc


def make_windows(n_tokens):
    """Start positions of overlapping windows that cover the whole document."""

    if n_tokens == 0:
        return []

    if n_tokens <= WINDOW:
        return [0]

    window_starts = list(range(0, n_tokens - WINDOW + 1, STRIDE))

    if window_starts[-1] != n_tokens - WINDOW:
        window_starts.append(n_tokens - WINDOW)

    return window_starts


start_time = time.time()

train_docs = [make_doc(i, t, gold[f][i]) for f in TRAIN_FOLDS for i, t in texts[f].items()]
val_docs = [make_doc(i, t, gold[f][i]) for f in VAL_FOLDS for i, t in texts[f].items()]
test_docs = [make_doc(i, t) for i, t in test_texts.items()]


# ------------------------------------------------------------
# Word-shape vocabulary (from the training documents; 0 = padding, 1 = unknown)
# ------------------------------------------------------------

shape_counts = Counter(s for d in train_docs for s in d["shapes"])

SHAPE2ID = {"<pad>": 0, "<unk>": 1}

for shape, count in shape_counts.most_common():
    if count >= 2:
        SHAPE2ID[shape] = len(SHAPE2ID)

for doc in train_docs + val_docs + test_docs:
    doc["shape_ids"] = np.array([SHAPE2ID.get(s, 1) for s in doc["shapes"]], dtype=np.int64)


# the token ids must match tokenizer.encode exactly
mismatch = 0

for doc in (train_docs + val_docs + test_docs)[:200]:
    if tok_module.normalize_text(doc["text"]) == doc["text"]:
        mismatch += doc["ids"].tolist() != tokenizer.encode(doc["text"])

assert mismatch == 0, f"{mismatch} documents differ from tokenizer.encode"

print(f"Tokenized in {time.time() - start_time:.1f}s | word shapes: {len(SHAPE2ID)}")

for name, docs in (("train", train_docs), ("val", val_docs), ("test", test_docs)):

    n_tokens = sum(len(d["ids"]) for d in docs)
    n_words = sum(len(d["first_token"]) for d in docs)
    n_windows = sum(len(make_windows(len(d["ids"]))) for d in docs)

    print(f"{name:5s}: {len(docs):4d} docs | {n_tokens:>9,} tokens | {n_words:>9,} words | {n_windows:>5,} windows")

print("Gold entities dropped while tagging (overlap / no word):", sum(d["dropped"] for d in train_docs + val_docs))

label_counts = Counter(l for f in TRAIN_FOLDS for spans in gold[f].values() for _, _, l in spans)

print("\nEntities per label in the training folds:")
for label in LABELS:
    print(f"  {label:10s} {label_counts[label]:6d}")

Tokenized in 9.5s | word shapes: 121
train:   35 docs |   462,556 tokens |   223,765 words | 1,211 windows
val  :   35 docs |   468,040 tokens |   225,985 words | 1,224 windows
test :   35 docs |   494,497 tokens |   239,728 words | 1,293 windows
Gold entities dropped while tagging (overlap / no word): 0

Entities per label in the training folds:
  APP           207
  RESP          191
  A.COUNSEL      70
  R.COUNSEL      87
  JUDGE         123
  WIT            67
  AUTH          535
  COURT         455
  STAT          525
  PREC          624
  DATE          813
  CASENO        423


### CELL 8 · Offset self-test (run before training!)

**In simple words:** A check without any model: turn the gold entities into word tags and back again. The score is the best any word-level tagger could ever reach here.

**How it works:** Expected ≈ 0.99 strict macro-F1; a clearly lower value would mean entity boundaries fall inside words (an offset bug).

In [26]:
# ============================================================
# CELL 8 — OFFSET SELF-TEST  (run before training!)
#
# gold entities -> word BIO tags -> spans back -> must equal gold.
# The score is the BEST F1 any model could ever reach with
# word-level tagging. If it is clearly below 1.0, some entity
# boundaries fall inside words.
# ============================================================

check_docs = val_docs if val_docs else train_docs
check_gold = {}
check_pred = {}

for doc in check_docs:

    check_gold[doc["id"]] = doc["spans"]
    check_pred[doc["id"]] = tags_to_spans(doc["word_tags"], doc["text"], doc["word_starts"], doc["word_ends"])

macro, per_label = strict_scores(check_gold, check_pred)

print(f"Round-trip (gold -> tags -> spans) strict macro-F1: {macro['f1']:.4f}")

for label, scores in per_label.items():
    if scores["f1"] < 0.999:
        print(f"  {label:10s} F1={scores['f1']:.4f}  (support {scores['support']})")

if macro["f1"] < 0.5:
    raise RuntimeError("Offsets look wrong (round-trip F1 < 0.5): check the end-offset convention and the sample in CELL 5.")

if macro["f1"] < 0.98:
    print("WARNING: some entity boundaries are inside words, so this is the maximum possible F1 for this setup.")
else:
    print("✅ Offsets are exact.")

CEILING_F1 = macro["f1"]

Round-trip (gold -> tags -> spans) strict macro-F1: 0.9938
  APP        F1=0.9900  (support 200)
  RESP       F1=0.9895  (support 190)
  AUTH       F1=0.9957  (support 468)
  COURT      F1=0.9785  (support 558)
  STAT       F1=0.9823  (support 735)
  PREC       F1=0.9893  (support 559)
✅ Offsets are exact.


### CELL 9 · Batches

**In simple words:** Group windows into training batches.

**How it works:** The model reads `WINDOW` tokens and predicts a tag for every word whose first token is in the window.

In [27]:
# ============================================================
# CELL 9 — BATCHES
#
# A window = WINDOW consecutive tokens. The model reads the tokens,
# then predicts one tag for every word whose FIRST token is inside
# the window.
# ============================================================

def window_words(doc, s):
    """Index range [w0, w1) of the words whose first token lies in the window starting at token s."""

    n = min(WINDOW, len(doc["ids"]) - s)

    w0 = int(np.searchsorted(doc["first_token"], s, side="left"))
    w1 = int(np.searchsorted(doc["first_token"], s + n, side="left"))

    return n, w0, w1


def make_batch(docs, items):
    """
    items = list of (doc_index, window_start).
    Returns a dict of tensors:
        ids, mask            [B, T]   tokens
        word_pos, word_mask  [B, W]   position of each word's first token in the window
        shape_ids            [B, W]
        flags                [B, W, NUM_FLAGS]
        labels               [B, W]   word tags (-1 = no label)
    """

    sizes = [window_words(docs[d], s) for d, s in items]

    T = max(n for n, _, _ in sizes)
    W = max(max(w1 - w0 for _, w0, w1 in sizes), 1)
    B = len(items)

    batch = {
        "ids": torch.full((B, T), PAD_ID, dtype=torch.long),
        "mask": torch.zeros((B, T), dtype=torch.bool),
        "word_pos": torch.zeros((B, W), dtype=torch.long),
        "word_mask": torch.zeros((B, W), dtype=torch.bool),
        "shape_ids": torch.zeros((B, W), dtype=torch.long),
        "flags": torch.zeros((B, W, NUM_FLAGS), dtype=torch.float32),
        "labels": torch.full((B, W), -1, dtype=torch.long),
    }

    for row, ((d, s), (n, w0, w1)) in enumerate(zip(items, sizes)):

        doc = docs[d]
        k = w1 - w0

        batch["ids"][row, :n] = torch.from_numpy(doc["ids"][s:s + n])
        batch["mask"][row, :n] = True

        batch["word_pos"][row, :k] = torch.from_numpy(doc["first_token"][w0:w1] - s)
        batch["word_mask"][row, :k] = True
        batch["shape_ids"][row, :k] = torch.from_numpy(doc["shape_ids"][w0:w1])
        batch["flags"][row, :k] = torch.from_numpy(doc["flags"][w0:w1])

        if doc["word_tags"] is not None and k > 0:

            tags = doc["word_tags"][w0:w1].copy()

            # window starts inside an entity: its first word becomes "B-" (an "I-" cannot start a sequence)
            if TAG_PARTS[tags[0]][0] == "I":
                tags[0] -= 1                                        # I-X -> B-X  (B-X is always the id before I-X)

            batch["labels"][row, :k] = torch.from_numpy(tags)

    return batch


def window_items(docs):
    """All (doc_index, window_start) pairs that contain at least one word."""

    return [
        (d, s)
        for d, doc in enumerate(docs)
        for s in make_windows(len(doc["ids"]))
        if window_words(doc, s)[2] > window_words(doc, s)[1]
    ]


example = make_batch(train_docs, window_items(train_docs)[:4])

print({k: tuple(v.shape) for k, v in example.items()})
print("Training windows:", len(window_items(train_docs)))

{'ids': (4, 512), 'mask': (4, 512), 'word_pos': (4, 261), 'word_mask': (4, 261), 'shape_ids': (4, 261), 'flags': (4, 261, 6), 'labels': (4, 261)}
Training windows: 1211


### CELL 10 · Models

**In simple words:** The two networks of the distillation run — the **teacher** (Architecture-4, partly fine-tuned) and the **student** (Architecture-2 encoder) — with the same word-tagging head.

**How it works:** first-token vector of every word + word shape + flags → BiLSTM → CRF. The distillation loss compares the teacher's and the student's tag probabilities (temperature T = 2).

In [28]:
# ============================================================
# CELL 10 — MODELS
#
#   backbone (Architecture-4 TEACHER  or  Architecture-2 STUDENT)
#     -> first-token vector of every word + word shape + flags
#     -> BiLSTM -> CRF                          (same head as E1/E2)
#
# Layer names match the Phase-1 notebooks, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-2 ENCODER (student backbone)
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q, k, v = self.split_heads(self.q_proj(x)), self.split_heads(self.k_proj(x)), self.split_heads(self.v_proj(x))

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):
        x = x + self.self_attention(self.norm1(x), attn_mask)
        return x + self.feed_forward(self.norm2(x))


class Arch2Encoder(nn.Module):
    """Architecture-2 encoder: bidirectional, learned positions (STUDENT)."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = PositionalEmbedding(cfg["src_len"], cfg["d_model"], cfg["dropout"])
        self.layers = nn.ModuleList([
            EncoderBlock(cfg["d_model"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.final_norm = nn.LayerNorm(cfg["d_model"])

    def forward(self, input_ids, mask):

        attn_mask = mask[:, None, None, :]                    # padding is never attended to
        x = self.position_embedding(self.token_embedding(input_ids))

        for layer in self.layers:
            x = layer(x, attn_mask)

        return self.final_norm(x)


# ============================================================
# Architecture-4 HYBRID PREFIX-LM (teacher backbone)
# For NER the prefix = the whole window, so every token goes
# through the bidirectional branch AND the causal branch.
# ============================================================

class HybridRMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (self.weight * (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps))).to(x.dtype)


class HybridAttention(nn.Module):
    """Used twice per block: bidirectional (encoder branch) and causal (causal branch)."""

    def __init__(self, d_model, n_heads, dropout, causal):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.causal = causal

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask):

        B, T, D = x.shape
        q, k, v = (p(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
                   for p in (self.q_proj, self.k_proj, self.v_proj))

        p = self.dropout.p if self.training else 0.0

        if self.causal:      # padding is at the end, so real tokens never see it
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=p)

        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, D))


class GatedHybridFusion(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.gate_proj = nn.Linear(2 * d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, encoder_output, causal_output):
        gate = torch.sigmoid(self.gate_proj(torch.cat([encoder_output, causal_output], dim=-1)))
        return self.out_proj(gate * encoder_output + (1.0 - gate) * causal_output)


class HybridSwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class HybridTransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        d, h, p, eps = cfg["d_model"], cfg["n_heads"], cfg["dropout"], cfg["norm_eps"]

        self.encoder_norm = HybridRMSNorm(d, eps)
        self.causal_norm = HybridRMSNorm(d, eps)
        self.encoder_attention = HybridAttention(d, h, p, causal=False)
        self.causal_attention = HybridAttention(d, h, p, causal=True)
        self.hybrid_fusion = GatedHybridFusion(d)
        self.ffn_norm = HybridRMSNorm(d, eps)
        self.ffn = HybridSwiGLU(d, cfg["ffn_dim"], p)

    def forward(self, x, attn_mask):

        fused = self.hybrid_fusion(
            self.encoder_attention(self.encoder_norm(x), attn_mask),
            self.causal_attention(self.causal_norm(x), attn_mask),
        )
        x = x + fused

        return x + self.ffn(self.ffn_norm(x))


class Arch4Hybrid(nn.Module):
    """Architecture-4 HybridLM without the LM head (TEACHER)."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["block_size"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([HybridTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = HybridRMSNorm(cfg["d_model"], cfg["norm_eps"])

    def forward(self, input_ids, mask):

        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))

        attn_mask = mask[:, None, None, :]

        for block in self.blocks:
            x = block(x, attn_mask)

        return self.final_norm(x)


# ============================================================
# CRF + NER model (same head for teacher and student)
# ============================================================

class CRF(nn.Module):
    """Linear-chain CRF. Invalid BIO transitions are fixed to -10000."""

    def __init__(self, num_tags):
        super().__init__()

        self.transitions = nn.Parameter(torch.zeros(num_tags, num_tags))      # [previous, current]
        self.start = nn.Parameter(torch.zeros(num_tags))
        self.end = nn.Parameter(torch.zeros(num_tags))

        self.register_buffer("invalid", torch.from_numpy(~ALLOWED))
        self.register_buffer("invalid_start", torch.from_numpy(~START_ALLOWED))

    def scores(self):

        transitions = self.transitions.masked_fill(self.invalid, -10000.0)
        start = self.start.masked_fill(self.invalid_start, -10000.0)

        return transitions, start, self.end

    def loss(self, emissions, tags, mask):
        """Mean negative log-likelihood per word. emissions [B, W, C], tags [B, W], mask [B, W]."""

        transitions, start, end = self.scores()

        B, W, C = emissions.shape
        tags = tags.clamp(min=0)
        maskf = mask.float()

        # score of the gold tag sequence
        gold = start[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, :1]).squeeze(1)

        for i in range(1, W):
            step = transitions[tags[:, i - 1], tags[:, i]] + emissions[:, i].gather(1, tags[:, i:i + 1]).squeeze(1)
            gold = gold + step * maskf[:, i]

        last_tag = tags.gather(1, (mask.sum(1) - 1).clamp(min=0).unsqueeze(1)).squeeze(1)
        gold = gold + end[last_tag]

        # log-sum of the scores of ALL tag sequences (forward algorithm)
        alpha = start + emissions[:, 0]

        for i in range(1, W):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + transitions.unsqueeze(0), dim=1) + emissions[:, i]
            alpha = torch.where(mask[:, i:i + 1], nxt, alpha)

        log_z = torch.logsumexp(alpha + end, dim=1)

        return (log_z - gold).sum() / mask.sum()


def distillation_loss(student_emissions, teacher_emissions, mask):
    """T^2 * KL(teacher || student) on the per-word tag distributions (Hinton et al., 2015)."""

    T = KD_TEMPERATURE

    student_log_probs = F.log_softmax(student_emissions / T, dim=-1)
    teacher_probs = F.softmax(teacher_emissions / T, dim=-1)

    kl = (teacher_probs * (torch.log(teacher_probs + 1e-9) - student_log_probs)).sum(-1)     # [B, W]

    return (kl * mask).sum() / mask.sum().clamp(min=1.0) * T * T


class NERModel(nn.Module):

    def __init__(self, backbone, d_model, num_shapes):
        super().__init__()

        self.backbone = backbone

        self.shape_embedding = nn.Embedding(num_shapes, SHAPE_DIM, padding_idx=0)
        self.dropout = nn.Dropout(HEAD_DROPOUT)
        self.lstm = nn.LSTM(d_model + SHAPE_DIM + NUM_FLAGS, LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.emission = nn.Linear(2 * LSTM_HIDDEN, NUM_TAGS)
        self.crf = CRF(NUM_TAGS)

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def emissions(self, batch):
        """Tag scores for every word in the batch: [B, W, NUM_TAGS] (float32)."""

        hidden = self.backbone(batch["ids"], batch["mask"])

        index = batch["word_pos"].unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        word_vectors = hidden.gather(1, index)

        x = torch.cat([word_vectors.float(), self.shape_embedding(batch["shape_ids"]), batch["flags"]], dim=-1)
        x = self.dropout(x)

        lengths = batch["word_mask"].sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))

        return self.emission(self.dropout(out)).float()

    def loss(self, batch):

        with autocast():
            emissions = self.emissions(batch)

        emissions = emissions.float()
        crf_loss = self.crf.loss(emissions, batch["labels"], batch["word_mask"])

        if "teacher" not in batch:                        # teacher / no distillation
            return crf_loss

        kd_mask = batch["word_mask"].float()

        if KD_ONLY_WHERE_TEACHER_CORRECT:
            teacher_tags = batch["teacher"].argmax(dim=-1)
            kd_mask = kd_mask * (teacher_tags == batch["labels"]).float()

        kd_loss = distillation_loss(emissions, batch["teacher"], kd_mask)

        return (1 - KD_ALPHA) * crf_loss + KD_ALPHA * kd_loss

### CELL 11 · Load both Phase-1 checkpoints + model builders

**In simple words:** Load the teacher's and the student's Phase-1 checkpoints and define how each network is built.

In [29]:
# ============================================================
# CELL 11 — LOAD BOTH PHASE-1 CHECKPOINTS + MODEL BUILDERS
# ============================================================

def find_checkpoint(path, file_name, required_key):
    """Use `path` if it exists, else search /kaggle/input for `file_name` whose config has `required_key`."""

    if Path(path).exists():
        return Path(path)

    for candidate in sorted(Path("/kaggle/input").rglob(file_name)):
        try:
            ckpt = torch.load(candidate, map_location="cpu", weights_only=False)
            config = ckpt.get("model_config", ckpt.get("config", {}))
            if required_key in config:
                print("Found:", candidate)
                return candidate
        except Exception:
            pass

    raise FileNotFoundError(f"{path} not found and no matching {file_name} under /kaggle/input")


def clean_state(state, keep_prefix=""):
    """Strip torch.compile prefixes, drop the LM head, keep keys under `keep_prefix`."""

    out = {}

    for key, value in state.items():
        key = key.replace("_orig_mod.", "")
        if key.startswith("lm_head.") or not key.startswith(keep_prefix):
            continue
        out[key[len(keep_prefix):]] = value

    return out


# ---------------- teacher: Architecture-4 ----------------
TEACHER_CHECKPOINT = find_checkpoint(TEACHER_CHECKPOINT, "best.pt", "prefix_length")
ckpt = torch.load(TEACHER_CHECKPOINT, map_location="cpu", weights_only=False)
TEACHER_CONFIG = dict(ckpt["model_config"])
TEACHER_WEIGHTS = clean_state(ckpt["model_state_dict"])
TEACHER_CONFIG.setdefault("norm_eps", 1e-5)

# ---------------- student: Architecture-2 encoder ----------------
STUDENT_CHECKPOINT = find_checkpoint(STUDENT_CHECKPOINT, "best_encoder_decoder.pt", "encoder_layers")
ckpt = torch.load(STUDENT_CHECKPOINT, map_location="cpu", weights_only=False)
STUDENT_CONFIG = dict(ckpt["config"])
STUDENT_WEIGHTS = clean_state(ckpt["model_state_dict"], keep_prefix="encoder.")

del ckpt

for name, cfg, max_len in (("teacher", TEACHER_CONFIG, TEACHER_CONFIG["block_size"]),
                           ("student", STUDENT_CONFIG, STUDENT_CONFIG["src_len"])):
    assert cfg["vocab_size"] == VOCAB_SIZE, f"{name} was trained with a different tokenizer"
    assert WINDOW <= max_len, f"WINDOW must be <= {max_len} for the {name}"

print("Teacher config:", {k: TEACHER_CONFIG[k] for k in ("d_model", "n_layers", "n_heads", "ffn_dim", "block_size", "prefix_length")})
print("Student config:", {k: STUDENT_CONFIG[k] for k in ("d_model", "encoder_layers", "n_heads", "ffn_dim", "src_len")})


def build_teacher():
    """Architecture-4 + head. Partial fine-tuning: only the top blocks, the final norm and the head train."""

    backbone = Arch4Hybrid(TEACHER_CONFIG)
    backbone.load_state_dict(TEACHER_WEIGHTS, strict=True)

    for p in backbone.parameters():
        p.requires_grad = False

    for module in list(backbone.blocks[-TEACHER_TRAIN_TOP_BLOCKS:]) + [backbone.final_norm]:
        for p in module.parameters():
            p.requires_grad = True

    return NERModel(backbone, TEACHER_CONFIG["d_model"], len(SHAPE2ID)).to(DEVICE)


def build_student():
    """Architecture-2 encoder + head. Full fine-tuning."""

    backbone = Arch2Encoder(STUDENT_CONFIG)
    backbone.load_state_dict(STUDENT_WEIGHTS, strict=True)

    return NERModel(backbone, STUDENT_CONFIG["d_model"], len(SHAPE2ID)).to(DEVICE)


def count(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# parameter counts (built on CPU just to count)
_t = NERModel(Arch4Hybrid(TEACHER_CONFIG), TEACHER_CONFIG["d_model"], len(SHAPE2ID))
for p in _t.backbone.parameters():
    p.requires_grad = False
for m in list(_t.backbone.blocks[-TEACHER_TRAIN_TOP_BLOCKS:]) + [_t.backbone.final_norm]:
    for p in m.parameters():
        p.requires_grad = True
_s = NERModel(Arch2Encoder(STUDENT_CONFIG), STUDENT_CONFIG["d_model"], len(SHAPE2ID))

TEACHER_PARAMS, TEACHER_TRAINABLE = count(_t)
STUDENT_PARAMS, STUDENT_TRAINABLE = count(_s)
del _t, _s

print(f"\nTeacher (E4): {TEACHER_PARAMS:,} parameters, {TEACHER_TRAINABLE:,} trainable "
      f"({TEACHER_TRAINABLE / TEACHER_PARAMS:.1%})")
print(f"Student (E5): {STUDENT_PARAMS:,} parameters, all trainable "
      f"-> {TEACHER_PARAMS / STUDENT_PARAMS:.1f}x smaller than the teacher")

Found: /kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-4/model4_checkpoints/best.pt
Found: /kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt
Teacher config: {'d_model': 768, 'n_layers': 16, 'n_heads': 12, 'ffn_dim': 2048, 'block_size': 2048, 'prefix_length': 512}
Student config: {'d_model': 768, 'encoder_layers': 6, 'n_heads': 12, 'ffn_dim': 3072, 'src_len': 1024}

Teacher (E4): 207,692,508 parameters, 91,868,892 trainable (44.2%)
Student (E5): 70,090,716 parameters, all trainable -> 3.0x smaller than the teacher


### CELL 12 · Prediction + evaluation

**In simple words:** Predict the entities of whole documents and score them.

**How it works:** Scores of overlapping windows are averaged per word, the CRF's Viterbi picks the best valid tag sequence for the whole document, tags become character spans, and strict macro-F1 compares them with gold.

In [30]:
# ============================================================
# CELL 12 — PREDICTION + EVALUATION
#
# Every window gives tag scores for its words. Words that appear in
# several windows get the average score. Then the CRF Viterbi picks
# the best valid tag sequence for the whole document.
# ============================================================

@torch.no_grad()
def predict_word_scores(model, docs):
    """Tag scores of every word (averaged over the windows it appears in) + the CRF scores."""

    model.eval()

    score_sum = [np.zeros((len(d["first_token"]), NUM_TAGS), dtype=np.float32) for d in docs]
    counts = [np.zeros(len(d["first_token"]), dtype=np.float32) for d in docs]

    items = window_items(docs)

    for b in range(0, len(items), EVAL_BATCH_SIZE):

        batch_items = items[b:b + EVAL_BATCH_SIZE]
        batch = {k: v.to(DEVICE) for k, v in make_batch(docs, batch_items).items()}

        with autocast():
            emissions = model.emissions(batch)

        emissions = emissions.float().cpu().numpy()

        for row, (d, s) in enumerate(batch_items):

            _, w0, w1 = window_words(docs[d], s)

            score_sum[d][w0:w1] += emissions[row, :w1 - w0]
            counts[d][w0:w1] += 1

    crf_scores = tuple(t.detach().float().cpu().numpy() for t in model.crf.scores())

    model.train()

    word_scores = [score_sum[d] / np.maximum(counts[d], 1)[:, None] for d in range(len(docs))]

    return word_scores, crf_scores


def decode_spans(docs, word_scores, crf_scores, o_bias=0.0):
    """CRF Viterbi per document. o_bias > 0 lowers the "O" score -> more entities (higher recall)."""

    transitions, start, end = crf_scores
    predictions = {}

    for doc, scores in zip(docs, word_scores):

        scores = scores.copy()
        scores[:, 0] -= o_bias

        word_tags = viterbi(scores, transitions, start, end)

        predictions[doc["id"]] = tags_to_spans(word_tags, doc["text"], doc["word_starts"], doc["word_ends"])

    return predictions


def predict_spans(model, docs, o_bias=0.0):
    """Returns {doc_id: [[start, end, label], ...]}"""

    word_scores, crf_scores = predict_word_scores(model, docs)

    return decode_spans(docs, word_scores, crf_scores, o_bias)


# ------------------------------------------------------------
# Recall tuning: the model predicts too few entities (recall < precision).
# Try a few penalties on the "O" score on the VALIDATION documents and
# keep the best one for the test set (no retraining needed).
# ------------------------------------------------------------

O_BIAS_CANDIDATES = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]


def tune_o_bias(model, docs):

    word_scores, crf_scores = predict_word_scores(model, docs)
    gold_spans = {d["id"]: d["spans"] for d in docs}

    table = {
        bias: strict_scores(gold_spans, decode_spans(docs, word_scores, crf_scores, bias))[0]
        for bias in O_BIAS_CANDIDATES
    }

    best = max(table, key=lambda bias: table[bias]["f1"])

    print("O-penalty   val strict macro  F1 /   P   /   R")

    for bias, m in table.items():
        mark = "   <- best" if bias == best else ""
        print(f"  {bias:4.1f}               {m['f1']:.4f} / {m['precision']:.4f} / {m['recall']:.4f}{mark}")

    return best


def evaluate(model, docs):

    gold_spans = {doc["id"]: doc["spans"] for doc in docs}

    return strict_scores(gold_spans, predict_spans(model, docs))


# ------------------------------------------------------------
# Document-level propagation
#
# Judgments repeat names: a party introduced in the header
# ("JAGE RAM ... Appellant") appears again and again in the body.
# For the labels in `labels`:
#   1. the same string gets its majority label everywhere in the document
#   2. every other occurrence of a predicted entity string is added
# ------------------------------------------------------------

import re


def propagate_entities(text, spans, labels):

    votes = {}

    for s, e, l in spans:
        votes.setdefault(text[s:e], Counter())[l] += 1

    majority = {}

    for string, counts in votes.items():

        label = counts.most_common(1)[0][0]

        if label in labels and len(string) >= 3 and any(ch.isalnum() for ch in string):
            majority[string] = label

    # 1. consistent labels
    output = [[s, e, majority.get(text[s:e], l)] for s, e, l in spans]

    covered = np.zeros(len(text) + 1, dtype=bool)

    for s, e, _ in output:
        covered[s:e] = True

    # 2. add the occurrences the model missed (longest strings first, no overlaps)
    for string, label in sorted(majority.items(), key=lambda item: -len(item[0])):

        for m in re.finditer(r"(?<!\w)" + re.escape(string) + r"(?!\w)", text):

            if not covered[m.start():m.end()].any():
                output.append([m.start(), m.end(), label])
                covered[m.start():m.end()] = True

    return sorted(output)


def propagate_all(predictions, texts_by_id, labels):
    """Apply propagate_entities to every document of a prediction dict."""

    return {doc_id: propagate_entities(texts_by_id[doc_id], spans, labels) for doc_id, spans in predictions.items()}

### CELL 13 · Training, teacher targets, submission helpers

**In simple words:** Helpers shared by teacher and student: the training loop, the teacher's scores for every window, reports and writing a submission.

In [31]:
# ============================================================
# CELL 13 — TRAINING, TEACHER TARGETS, SUBMISSION HELPERS
# ============================================================

def make_optimizer(param_groups, total_steps):
    """AdamW + linear warmup / linear decay. param_groups: list of {"params", "lr", "weight_decay"}."""

    optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

    return optimizer, set_lr


def teacher_param_groups(model):
    """Layer-wise learning rates: the top block gets LR_TEACHER_BLOCKS,
    every block below it TEACHER_LAYER_LR_DECAY times less (frozen blocks: none)."""

    trainable_blocks = [b for b in model.backbone.blocks if any(p.requires_grad for p in b.parameters())]

    groups = [{"params": list(model.backbone.final_norm.parameters()), "lr": LR_TEACHER_BLOCKS, "weight_decay": 0.0}]
    lr = LR_TEACHER_BLOCKS

    for block in reversed(trainable_blocks):                  # top block first
        groups.append({"params": list(block.parameters()), "lr": lr, "weight_decay": WEIGHT_DECAY})
        lr *= TEACHER_LAYER_LR_DECAY

    groups.append({"params": model.head_parameters(), "lr": LR_TEACHER_HEAD, "weight_decay": 0.0})

    return groups


def student_param_groups(model):
    decay = [p for p in model.backbone.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.backbone.parameters() if p.dim() < 2]
    return [{"params": decay, "lr": LR_STUDENT_ENCODER, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "lr": LR_STUDENT_ENCODER, "weight_decay": 0.0},
            {"params": model.head_parameters(), "lr": LR_STUDENT_HEAD, "weight_decay": 0.0}]


def add_teacher_targets(batch, docs, items, teacher_cache):
    """Adds batch["teacher"] [B, W, NUM_TAGS]: the cached teacher scores of every word."""

    teacher = torch.zeros(batch["labels"].shape + (NUM_TAGS,), dtype=torch.float32)

    for row, (d, s) in enumerate(items):
        scores = teacher_cache[(docs[d]["id"], s)]
        teacher[row, :len(scores)] = torch.from_numpy(scores)

    batch["teacher"] = teacher
    return batch


def train_model(model, docs, epochs, param_groups, batch_size,
                val_docs=None, save_path=None, patience=None, teacher_cache=None):
    """Trains `model`. With val_docs: keeps the best epoch (strict macro-F1).
    With teacher_cache: knowledge distillation (CRF loss + KL to the teacher)."""

    items = window_items(docs)
    steps_per_epoch = math.ceil(len(items) / batch_size)

    optimizer, set_lr = make_optimizer(param_groups, steps_per_epoch * epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    rng = np.random.default_rng(SEED)
    history = []
    best_f1, best_epoch, step = -1.0, 0, 0
    start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        order = rng.permutation(len(items))
        epoch_loss = 0.0

        for b in range(0, len(order), batch_size):

            batch_items = [items[i] for i in order[b:b + batch_size]]
            batch = make_batch(docs, batch_items)

            if teacher_cache is not None:
                batch = add_teacher_targets(batch, docs, batch_items, teacher_cache)

            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            set_lr(step)
            loss = model.loss(batch)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            step += 1
            epoch_loss += loss.item()

        row = {"epoch": epoch, "train_loss": epoch_loss / steps_per_epoch,
               "train_f1": evaluate(model, docs[:TRAIN_EVAL_DOCS])[0]["f1"]}
        line = f"Epoch {epoch:2d}/{epochs} | loss={row['train_loss']:.4f} train F1={row['train_f1']:.4f}"

        if val_docs:
            macro, _ = evaluate(model, val_docs)
            row.update(val_f1=macro["f1"], val_precision=macro["precision"], val_recall=macro["recall"])
            line += f" | val F1={macro['f1']:.4f} P={macro['precision']:.4f} R={macro['recall']:.4f}"

            if macro["f1"] > best_f1:
                best_f1, best_epoch = macro["f1"], epoch
                torch.save(model.state_dict(), save_path)
                line += " ⭐ best"

        history.append(row)
        print(line + f" | {(time.time() - start) / 60:.1f} min", flush=True)

        if val_docs and patience and epoch - best_epoch >= patience:
            print(f"No improvement for {patience} epochs -> early stop.")
            break

    if not val_docs and save_path:
        torch.save(model.state_dict(), save_path)
        best_epoch = epochs

    peak_gb = torch.cuda.max_memory_allocated() / 1024 ** 3 if DEVICE.type == "cuda" else 0.0

    return {"history": history, "best_epoch": best_epoch, "best_f1": best_f1,
            "minutes": (time.time() - start) / 60, "peak_gpu_gb": peak_gb}


@torch.no_grad()
def teacher_targets(model, docs):
    """{(doc_id, window_start): teacher scores [n_words_in_window, NUM_TAGS]} for every training window."""

    model.eval()
    cache = {}
    items = window_items(docs)

    for b in range(0, len(items), EVAL_BATCH_SIZE):

        batch_items = items[b:b + EVAL_BATCH_SIZE]
        batch = {k: v.to(DEVICE) for k, v in make_batch(docs, batch_items).items()}

        with autocast():
            emissions = model.emissions(batch).float().cpu().numpy()

        for row, (d, s) in enumerate(batch_items):
            _, w0, w1 = window_words(docs[d], s)
            cache[(docs[d]["id"], s)] = emissions[row, :w1 - w0].copy()

    model.train()
    return cache


def report(model, docs, name):
    """Recall tuning + document-level propagation (both chosen on `docs` = validation),
    per-label table, timing. Returns (macro, per_label, seconds, post) where `post`
    holds the chosen post-processing for the test set."""

    o_bias = tune_o_bias(model, docs)

    start = time.time()
    predictions = predict_spans(model, docs, o_bias)
    seconds = time.time() - start

    gold_spans = {d["id"]: d["spans"] for d in docs}
    texts_by_id = {d["id"]: d["text"] for d in docs}

    _, per_label_raw = strict_scores(gold_spans, predictions)

    propagate_labels = set()

    if DOC_PROPAGATION:
        _, per_label_all = strict_scores(gold_spans, propagate_all(predictions, texts_by_id, set(LABELS)))
        propagate_labels = {l for l in per_label_all if per_label_all[l]["f1"] > per_label_raw.get(l, {"f1": 0.0})["f1"]}
        predictions = propagate_all(predictions, texts_by_id, propagate_labels)

    macro, per_label = strict_scores(gold_spans, predictions)

    print(f"\n{name}: strict macro F1={macro['f1']:.4f}  P={macro['precision']:.4f}  R={macro['recall']:.4f}"
          f"  | O-penalty {o_bias} | propagation for {sorted(propagate_labels)}"
          f"  | prediction time {seconds:.1f}s for {len(docs)} docs\n")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'support':>8s}")
    for label, s in per_label.items():
        print(f"{label:10s} {s['precision']:7.3f} {s['recall']:7.3f} {s['f1']:7.3f} {s['support']:8d}")

    # files for the official local evaluator
    with open(OUTPUT_DIR / f"val_prediction_{name}.json", "w", encoding="utf-8") as f:
        json.dump({k: [[s, e - END_SHIFT, l] for s, e, l in v] for k, v in predictions.items()}, f, ensure_ascii=False)

    return macro, per_label, seconds, {"o_bias": o_bias, "propagate_labels": sorted(propagate_labels)}


def write_submission(model, name, post):
    """Predict the test set (with the post-processing chosen on validation)
    -> OUTPUT_DIR/<name>/prediction.json + submission.zip (validated)."""

    folder = OUTPUT_DIR / name
    folder.mkdir(exist_ok=True)

    submission = predict_spans(model, test_docs, post["o_bias"])

    if post["propagate_labels"]:
        submission = propagate_all(submission, test_texts, set(post["propagate_labels"]))

    assert set(submission) == set(test_texts), "prediction IDs must equal the official test IDs"
    for doc_id, spans in submission.items():
        for s, e, l in spans:
            assert 0 <= s < e <= len(test_texts[doc_id]) and l in LABELS, (doc_id, s, e, l)

    output = {k: [[s, e - END_SHIFT, l] for s, e, l in v] for k, v in submission.items()}

    with open(folder / "prediction.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)

    with zipfile.ZipFile(folder / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(folder / "prediction.json", arcname="prediction.json")

    n = sum(len(v) for v in output.values())
    print(f"✅ {name}: {len(output)} test documents, {n} entities -> {folder / 'submission.zip'}")


def free_gpu_memory():
    """Call AFTER setting the model variable to None."""
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

### CELL 14 · E4 stage A: teacher (Architecture-4, partial fine-tuning)

**In simple words:** Train the **teacher**: Architecture-4 with its top blocks unfrozen (partial fine-tuning), fold 1 → checked on fold 2.

In [32]:
# ============================================================
# CELL 14 — E4 STAGE A: TEACHER (Architecture-4, partial fine-tuning)
# ============================================================

print("=" * 90)
print(f"E4 TEACHER | train {TRAIN_FOLDS} | validate {VAL_FOLDS} | top {TEACHER_TRAIN_TOP_BLOCKS} blocks + head")
print("=" * 90)

torch.manual_seed(SEED)
teacher = build_teacher()

teacher_a = train_model(
    teacher, train_docs, TEACHER_EPOCHS, teacher_param_groups(teacher), TEACHER_BATCH_SIZE,
    val_docs=val_docs, save_path=OUTPUT_DIR / "teacher_stageA.pt", patience=TEACHER_PATIENCE,
)

teacher.load_state_dict(torch.load(OUTPUT_DIR / "teacher_stageA.pt", map_location=DEVICE))
teacher_macro, teacher_per_label, teacher_seconds, teacher_post = report(teacher, val_docs, "E4_teacher")

print(f"\nbest epoch {teacher_a['best_epoch']} | {teacher_a['minutes']:.1f} min | peak GPU {teacher_a['peak_gpu_gb']:.2f} GB")

E4 TEACHER | train ['fold1'] | validate ['fold2'] | top 8 blocks + head
Epoch  1/20 | loss=0.3647 train F1=0.0000 | val F1=0.0000 P=0.0000 R=0.0000 ⭐ best | 1.9 min
Epoch  2/20 | loss=0.1113 train F1=0.0398 | val F1=0.0413 P=0.0731 R=0.0291 ⭐ best | 3.9 min
Epoch  3/20 | loss=0.0916 train F1=0.1171 | val F1=0.0928 P=0.1443 R=0.0751 ⭐ best | 5.8 min
Epoch  4/20 | loss=0.0780 train F1=0.1727 | val F1=0.1514 P=0.1769 R=0.1441 ⭐ best | 7.8 min
Epoch  5/20 | loss=0.0702 train F1=0.1881 | val F1=0.1569 P=0.2451 R=0.1323 ⭐ best | 9.7 min
Epoch  6/20 | loss=0.0655 train F1=0.2275 | val F1=0.1719 P=0.2175 R=0.1666 ⭐ best | 11.7 min
Epoch  7/20 | loss=0.0613 train F1=0.2330 | val F1=0.1671 P=0.2493 R=0.1391 | 13.6 min
Epoch  8/20 | loss=0.0579 train F1=0.3051 | val F1=0.2122 P=0.2671 R=0.1932 ⭐ best | 15.5 min
Epoch  9/20 | loss=0.0544 train F1=0.3246 | val F1=0.2111 P=0.3039 R=0.1845 | 17.4 min
Epoch 10/20 | loss=0.0524 train F1=0.3022 | val F1=0.2283 P=0.3260 R=0.1968 ⭐ best | 19.4 min
Epoch 1

### CELL 15 · E5 stage A: student (Architecture-2) with knowledge distillation

**In simple words:** Train the **student** (Architecture-2 encoder) to match both the gold tags and the teacher's tag probabilities.

**How it works:** The teacher's scores are computed once for every training window, then the teacher leaves the GPU. Loss = 0.5 · gold loss + 0.5 · T²·KL(teacher ‖ student), T = 2, used only on words the teacher labels correctly.

In [33]:
# ============================================================
# CELL 15 — E5 STAGE A: STUDENT (Architecture-2) WITH KNOWLEDGE DISTILLATION
# ============================================================

# 1. teacher scores for every training window (computed once), then free the teacher
start = time.time()
cache_a = teacher_targets(teacher, train_docs)
print(f"Teacher targets for {len(cache_a)} training windows in {time.time() - start:.0f}s")

teacher = None
free_gpu_memory()

# 2. train the student on gold tags + teacher scores
print("=" * 90)
print(f"E5 STUDENT (KD) | train {TRAIN_FOLDS} | validate {VAL_FOLDS} | alpha {KD_ALPHA}, T {KD_TEMPERATURE}")
print("=" * 90)

torch.manual_seed(SEED)
student = build_student()

student_a = train_model(
    student, train_docs, STUDENT_EPOCHS, student_param_groups(student), STUDENT_BATCH_SIZE,
    val_docs=val_docs, save_path=OUTPUT_DIR / "student_stageA.pt", patience=STUDENT_PATIENCE,
    teacher_cache=cache_a,
)

student.load_state_dict(torch.load(OUTPUT_DIR / "student_stageA.pt", map_location=DEVICE))
student_macro, student_per_label, student_seconds, student_post = report(student, val_docs, "E5_student")

print(f"\nbest epoch {student_a['best_epoch']} | {student_a['minutes']:.1f} min | peak GPU {student_a['peak_gpu_gb']:.2f} GB")

student = None
del cache_a
free_gpu_memory()

Teacher targets for 1211 training windows in 24s
E5 STUDENT (KD) | train ['fold1'] | validate ['fold2'] | alpha 0.5, T 2.0
Epoch  1/30 | loss=0.8084 train F1=0.0064 | val F1=0.0115 P=0.1038 R=0.0067 ⭐ best | 1.0 min
Epoch  2/30 | loss=0.1049 train F1=0.2838 | val F1=0.2167 P=0.2859 R=0.1837 ⭐ best | 2.0 min
Epoch  3/30 | loss=0.0637 train F1=0.4692 | val F1=0.2947 P=0.3724 R=0.2709 ⭐ best | 3.1 min
Epoch  4/30 | loss=0.0500 train F1=0.6327 | val F1=0.3380 P=0.4023 R=0.3073 ⭐ best | 4.1 min
Epoch  5/30 | loss=0.0421 train F1=0.6595 | val F1=0.3512 P=0.4156 R=0.3237 ⭐ best | 5.1 min
Epoch  6/30 | loss=0.0377 train F1=0.7532 | val F1=0.3356 P=0.4357 R=0.2966 | 6.1 min
Epoch  7/30 | loss=0.0338 train F1=0.8180 | val F1=0.3847 P=0.4432 R=0.3607 ⭐ best | 7.1 min
Epoch  8/30 | loss=0.0311 train F1=0.8705 | val F1=0.3764 P=0.4797 R=0.3333 | 8.2 min
Epoch  9/30 | loss=0.0289 train F1=0.8827 | val F1=0.3849 P=0.4757 R=0.3528 ⭐ best | 9.2 min
Epoch 10/30 | loss=0.0271 train F1=0.9157 | val F1=0.3

### CELL 16 · Teacher vs student (for the report) + save results

**In simple words:** Compare teacher and student — score, size and speed — for the report.

In [34]:
# ============================================================
# CELL 16 — TEACHER vs STUDENT (for the report) + SAVE RESULTS
# ============================================================

rows = [
    ("E4 teacher (Arch-4, partial FT)", TEACHER_PARAMS, TEACHER_TRAINABLE, teacher_macro["f1"],
     teacher_a["peak_gpu_gb"], teacher_a["minutes"], teacher_seconds),
    ("E5 student (Arch-2, KD)", STUDENT_PARAMS, STUDENT_TRAINABLE, student_macro["f1"],
     student_a["peak_gpu_gb"], student_a["minutes"], student_seconds),
]

print(f"{'model':34s} {'params':>12s} {'trainable':>12s} {'val F1':>7s} {'GPU GB':>7s} {'train min':>9s} {'predict s':>9s}")
for name, total, trainable, f1, gb, minutes, seconds in rows:
    print(f"{name:34s} {total:12,d} {trainable:12,d} {f1:7.4f} {gb:7.2f} {minutes:9.1f} {seconds:9.1f}")

print(f"\nStudent F1 = {student_macro['f1'] / max(teacher_macro['f1'], 1e-9):.0%} of the teacher's, "
      f"with {STUDENT_PARAMS / TEACHER_PARAMS:.0%} of its parameters "
      f"and {student_seconds / max(teacher_seconds, 1e-9):.0%} of its prediction time.")

if student_macro["f1"] > teacher_macro["f1"]:
    print("The student beats its teacher: the teacher is the bottleneck (see its train F1), "
          "and selective KD kept its mistakes out of the student.")
print("Compare E5 with E1 (same student, no teacher) to see what distillation adds.")

results = {
    "E4_teacher": {"method": "Architecture-4 hybrid, partial fine-tuning (top blocks + head)",
                   "top_blocks_trained": TEACHER_TRAIN_TOP_BLOCKS, "layer_lr_decay": TEACHER_LAYER_LR_DECAY,
                   "parameters": TEACHER_PARAMS,
                   "trainable": TEACHER_TRAINABLE, "val_macro": teacher_macro, "val_per_label": teacher_per_label,
                   "prediction_seconds": teacher_seconds, "post_processing": teacher_post, **teacher_a},
    "E5_student": {"method": "Architecture-2 encoder, full fine-tuning + knowledge distillation from E4",
                   "kd_alpha": KD_ALPHA, "kd_temperature": KD_TEMPERATURE,
                   "kd_only_where_teacher_correct": KD_ONLY_WHERE_TEACHER_CORRECT, "parameters": STUDENT_PARAMS,
                   "trainable": STUDENT_TRAINABLE, "val_macro": student_macro, "val_per_label": student_per_label,
                   "prediction_seconds": student_seconds, "post_processing": student_post, **student_a},
    "window": WINDOW, "stride": STRIDE, "round_trip_ceiling_f1": CEILING_F1,
}

with open(OUTPUT_DIR / "e4_e5_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", OUTPUT_DIR / "e4_e5_results.json")

model                                    params    trainable  val F1  GPU GB train min predict s
E4 teacher (Arch-4, partial FT)     207,692,508   91,868,892  0.3618    8.21      38.7      25.1
E5 student (Arch-2, KD)              70,090,716   70,090,716  0.4309    6.04      16.3       8.0

The student beats its teacher: the teacher is the bottleneck (see its train F1), and selective KD kept its mistakes out of the student.
Compare E5 with E1 (same student, no teacher) to see what distillation adds.

Saved: /kaggle/working/e4_e5_teacher_student/e4_e5_results.json


### CELL 17 · Stage B teacher (fold1 + fold2) -> E4 submission

**In simple words:** Retrain the teacher on fold 1 + fold 2 and write its own submission (E4).

In [35]:
# ============================================================
# CELL 17 — STAGE B TEACHER (fold1 + fold2) -> E4 SUBMISSION
# ============================================================

all_docs = train_docs + val_docs

torch.manual_seed(SEED)
teacher = build_teacher()

if FINAL_TRAIN_ON_ALL_FOLDS:
    print(f"E4 teacher stage B: {TRAIN_FOLDS + VAL_FOLDS} for {teacher_a['best_epoch']} epochs")
    train_model(teacher, all_docs, max(1, teacher_a["best_epoch"]), teacher_param_groups(teacher),
                TEACHER_BATCH_SIZE, save_path=OUTPUT_DIR / "teacher_final.pt")
else:
    teacher.load_state_dict(torch.load(OUTPUT_DIR / "teacher_stageA.pt", map_location=DEVICE))

write_submission(teacher, "E4_teacher", teacher_post)      # E4 is done even if a later cell fails

# teacher scores on ALL training windows for the final student
cache_b = teacher_targets(teacher, all_docs if FINAL_TRAIN_ON_ALL_FOLDS else train_docs)

teacher = None
free_gpu_memory()

E4 teacher stage B: ['fold1', 'fold2'] for 19 epochs
Epoch  1/19 | loss=0.2801 train F1=0.0336 | 2.9 min
Epoch  2/19 | loss=0.0911 train F1=0.0971 | 5.8 min
Epoch  3/19 | loss=0.0732 train F1=0.1403 | 8.7 min
Epoch  4/19 | loss=0.0638 train F1=0.2557 | 11.7 min
Epoch  5/19 | loss=0.0579 train F1=0.2696 | 14.6 min
Epoch  6/19 | loss=0.0555 train F1=0.3378 | 17.5 min
Epoch  7/19 | loss=0.0516 train F1=0.3483 | 20.5 min
Epoch  8/19 | loss=0.0486 train F1=0.4315 | 23.4 min
Epoch  9/19 | loss=0.0467 train F1=0.4538 | 26.3 min
Epoch 10/19 | loss=0.0449 train F1=0.4888 | 29.2 min
Epoch 11/19 | loss=0.0423 train F1=0.4738 | 32.2 min
Epoch 12/19 | loss=0.0400 train F1=0.5300 | 35.1 min
Epoch 13/19 | loss=0.0386 train F1=0.5876 | 38.0 min
Epoch 14/19 | loss=0.0367 train F1=0.5946 | 41.0 min
Epoch 15/19 | loss=0.0348 train F1=0.6038 | 43.9 min
Epoch 16/19 | loss=0.0331 train F1=0.6336 | 46.9 min
Epoch 17/19 | loss=0.0314 train F1=0.6583 | 49.8 min
Epoch 18/19 | loss=0.0302 train F1=0.6621 | 52.7 

### CELL 18 · Stage B student (fold1 + fold2, distilled) -> E5 submission

**In simple words:** Retrain the student with distillation on fold 1 + fold 2 and write the distilled submission (E5).

In [36]:
# ============================================================
# CELL 18 — STAGE B STUDENT (fold1 + fold2, distilled) -> E5 SUBMISSION
# ============================================================

torch.manual_seed(SEED)
student = build_student()

if FINAL_TRAIN_ON_ALL_FOLDS:
    print(f"E5 student stage B: {TRAIN_FOLDS + VAL_FOLDS} for {student_a['best_epoch']} epochs (with teacher targets)")
    train_model(student, all_docs, max(1, student_a["best_epoch"]), student_param_groups(student),
                STUDENT_BATCH_SIZE, save_path=OUTPUT_DIR / "student_final.pt", teacher_cache=cache_b)
else:
    student.load_state_dict(torch.load(OUTPUT_DIR / "student_stageA.pt", map_location=DEVICE))

write_submission(student, "E5_student_kd", student_post)

print("\nSubmit:")
print("  E4 ->", OUTPUT_DIR / "E4_teacher" / "submission.zip")
print("  E5 ->", OUTPUT_DIR / "E5_student_kd" / "submission.zip")

E5 student stage B: ['fold1', 'fold2'] for 10 epochs (with teacher targets)
Epoch  1/10 | loss=0.4254 train F1=0.3495 | 1.7 min
Epoch  2/10 | loss=0.0677 train F1=0.5184 | 3.5 min
Epoch  3/10 | loss=0.0531 train F1=0.7009 | 5.3 min
Epoch  4/10 | loss=0.0457 train F1=0.7669 | 7.0 min
Epoch  5/10 | loss=0.0409 train F1=0.8255 | 8.7 min
Epoch  6/10 | loss=0.0376 train F1=0.8663 | 10.5 min
Epoch  7/10 | loss=0.0351 train F1=0.8938 | 12.2 min
Epoch  8/10 | loss=0.0332 train F1=0.9063 | 14.0 min
Epoch  9/10 | loss=0.0320 train F1=0.9066 | 15.7 min
Epoch 10/10 | loss=0.0311 train F1=0.9174 | 17.4 min
✅ E5_student_kd: 35 test documents, 3897 entities -> /kaggle/working/e4_e5_teacher_student/E5_student_kd/submission.zip

Submit:
  E4 -> /kaggle/working/e4_e5_teacher_student/E4_teacher/submission.zip
  E5 -> /kaggle/working/e4_e5_teacher_student/E5_student_kd/submission.zip
